# Problem 4: Programming with CVX
## 18-660/18-460 Optimization

This notebook contains the solution for the resource allocation optimization problem using `cvxpy`.

In [1]:
import cvxpy as cp
import numpy as np
import matplotlib.pyplot as plt

# Constants for the problem
n = 10  # Number of agents
D = 10  # Total resource budget
indices = np.arange(1, n + 1)
coefficients = -indices  # coefficients for f_i(x_i) = -ix_i

ModuleNotFoundError: No module named 'cvxpy'

### Part 1: Basic Resource Allocation

Objective: $\min \sum_{i=1}^n f_i(x_i)$ where $f_i(x_i) = -i x_i$

Subject to:
- $x_i \ge 0$
- $\sum x_i \le D$

In [ ]:
# Define variables
x = cp.Variable(n)

# Define objective and constraints
objective1 = cp.Minimize(coefficients @ x)
constraints1 = [x >= 0, cp.sum(x) <= D]

# Solve the problem
prob1 = cp.Problem(objective1, constraints1)
prob1.solve()

print(f"Status: {prob1.status}")
print(f"Optimal value: {prob1.value:.4f}")
print("Optimal allocation (x):")
for i in range(n):
    print(f"  x_{i+1} = {x.value[i]:.4f}")

# Plotting
plt.figure(figsize=(8, 5))
plt.bar(indices, x.value, color='steelblue', edgecolor='black')
plt.xlabel('Agent index i')
plt.ylabel('Allocated resource x_i')
plt.title('Part 1: Basic Resource Allocation')
plt.xticks(indices)
plt.grid(axis='y', alpha=0.3)
plt.show()

### Part 2: Fair Resource Allocation with Log Regularization

Objective: $\min \sum_{i=1}^n f_i(x_i) - \tau \sum_{i=1}^n \log x_i$

The log term acts as a "fairness regularizer" by penalizing zero allocations.

In [ ]:
tau_values = [0.1, 1.0, 5.0]
results = {}

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for idx, tau in enumerate(tau_values):
    # x_fair must be positive for log
    x_fair = cp.Variable(n, pos=True)
    
    # Define problem
    obj_fair = cp.Minimize(coefficients @ x_fair - tau * cp.sum(cp.log(x_fair)))
    const_fair = [cp.sum(x_fair) <= D]
    
    # Solve
    prob_fair = cp.Problem(obj_fair, const_fair)
    prob_fair.solve()
    
    results[tau] = x_fair.value.copy()
    
    # Display partial results
    print(f"tau = {tau} | optimal value = {prob_fair.value:.4f}")
    
    # Subplot
    axes[idx].bar(indices, x_fair.value, color='coral', edgecolor='black')
    axes[idx].set_title(f'tau = {tau}')
    axes[idx].set_xlabel('Agent index i')
    axes[idx].set_xticks(indices)
    axes[idx].set_ylim(0, max(x_fair.value) * 1.1)

plt.suptitle('Part 2: Fair Resource Allocation (Log Regularization)')
plt.tight_layout()
plt.show()

### Comparison and Final Analysis

The following plot compares all four scenarios (no fairness vs different levels of fairness).

In [ ]:
plt.figure(figsize=(12, 6))
width = 0.2
colors = ['steelblue', 'lightcoral', 'coral', 'orangered']
scenarios = ["Part 1 (no fairness)", "tau=0.1", "tau=1.0", "tau=5.0"]
data = [x.value] + [results[t] for t in tau_values]

for i, d in enumerate(data):
    plt.bar(indices + (i - 1.5) * width, d, width, label=scenarios[i], color=colors[i], edgecolor='black')

plt.xlabel('Agent index i')
plt.ylabel('Allocated resource x_i')
plt.title('Comparison of Resource Allocation Strategies')
plt.xticks(indices)
plt.legend()
plt.grid(axis='y', alpha=0.3)
plt.show()

### Discussion

- **Utility vs. Fairness**: In Part 1, the optimal strategy is purely "greedy," giving all resources to agent 10. This maximizes utility but is completely unfair.
- **Log Barrier**: The $-\tau \sum \log x_i$ term prevents $x_i$ from becoming zero because the log function approaches $-\infty$ near zero. This forces a non-zero allocation to all agents.
- **Sensitivity to $\tau$**: As $\tau$ increases, the allocation moves away from the greedy vertex toward a more uniform distribution.